# Preprocessing AQMS & Health Outcomes

## Keputusan dan asumsi

| Topik | Keputusan |
|---|---|
| PM2.5/PM10 ppm (PSW-01, MDN-01, BJB-01) | `pm25_ug_m3`/`pm10_ug_m3` **unresolved (NULL)**. Kandidat `*_candidate_ug_m3 = ppm x 1000` disediakan dan diberi label asumsi. |
| Gas ppm (SO2, CO, O3, NO2) | Konversi per baris: `ug/m3 = ppm x MW x 1000 / (0.082057366 x T_K / 1 atm)`. Tekanan **diasumsikan 1 atm** karena tidak tersedia. Fallback suhu 25 C jika suhu invalid. |
| Missing value | Hanya blank yang diimputasi dengan mean per `sensor_id` + parameter dari nilai VALID. Sentinel/teks error tidak diimputasi. |
| Outlier | IQR 1.5 per sensor/parameter; hanya flag, tidak ada drop/clip/winsorization. |
| QC flag | Seluruh kategori dipertahankan; `qc_flag` tidak dihapus dan tidak dianggap setara. |
| Bridge site-hospital | **Tidak dibuat** karena tidak ada booklet/pasangan resmi. Tidak ada pairing berdasarkan kedekatan/nama kota. |
| Output | Folder `data/`; notebook dieksekusi penuh. |


## 1. Setup

In [ ]:
from pathlib import Path
import json
import os
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 80)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = Path(os.environ.get("AQMS_RAW_DIR", PROJECT_ROOT / "data" / "raw"))
BASE_DIR = RAW_DIR
OUTPUT_DIR = Path(os.environ.get("AQMS_OUTPUT_DIR", PROJECT_ROOT / "data"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

AQMS_FILES = sorted(BASE_DIR.glob("*_AQMS.csv"))
HEALTH_FILE = BASE_DIR / "HLT-01_health_outcomes_daily.csv"
SITES_FILE = BASE_DIR / "sites.csv"
SENSORS_FILE = BASE_DIR / "sensors.csv"
HOSPITALS_FILE = BASE_DIR / "hospitals.csv"
META_SENSORS_FILE = BASE_DIR / "metadata" / "metadata_sensors.csv"
META_DATES_FILE = BASE_DIR / "metadata" / "metadata_dates.csv"

POLLUTANTS = ["pm25", "pm10", "so2", "co", "o3", "no2"]
GAS_PARAMS = ["so2", "co", "o3", "no2"]
PM_PARAMS = ["pm25", "pm10"]
METEO_COLS = ["temperature_c", "humidity_pct", "wind_speed_ms", "wind_direction_deg"]

MOLECULAR_WEIGHT = {"so2": 64.066, "co": 28.010, "o3": 48.000, "no2": 46.0055}
R_ATM_L_PER_MOL_K = 0.082057366
P_ATM = 1.0
TEMP_FALLBACK_C = 25.0
PM_CANDIDATE_FACTOR = 1000.0

PLAUSIBLE_BOUNDS = {
    "pm25": (0.0, 2000.0),
    "pm10": (0.0, 3000.0),
    "so2": (0.0, 2000.0),
    "co": (0.0, 60000.0),
    "o3": (0.0, 2000.0),
    "no2": (0.0, 2000.0),
    "temperature_c": (-20.0, 60.0),
    "humidity_pct": (0.0, 100.0),
    "wind_speed_ms": (0.0, 60.0),
    "wind_direction_deg": (0.0, 360.0),
}
PPM_BOUNDS = {
    "pm25": (0.0, 2.0),
    "pm10": (0.0, 3.0),
    "so2": (0.0, 5.0),
    "co": (0.0, 200.0),
    "o3": (0.0, 2.0),
    "no2": (0.0, 2.0),
}

print(f"Base directory : {BASE_DIR}")
print(f"Output folder  : {OUTPUT_DIR}")
print(f"File AQMS      : {len(AQMS_FILES)}")


In [ ]:
sites = pd.read_csv(SITES_FILE, dtype=str)
sensors = pd.read_csv(SENSORS_FILE, dtype=str)
hospitals = pd.read_csv(HOSPITALS_FILE, dtype=str)
meta_sensors = pd.read_csv(META_SENSORS_FILE, dtype=str)
meta_dates = pd.read_csv(META_DATES_FILE, dtype=str)
health_raw = pd.read_csv(HEALTH_FILE, dtype=str)

sensor_to_location = dict(zip(sensors["sensor_id"], sensors["location_id"]))
site_by_loc = sites.set_index("location_id").to_dict("index")

print("sites           :", sites.shape)
print("sensors         :", sensors.shape)
print("hospitals       :", hospitals.shape)
print("metadata_sensors:", meta_sensors.shape)
print("metadata_dates  :", meta_dates.shape)
print("health          :", health_raw.shape)

display(meta_sensors)
display(sites[["location_id", "station_code", "kota_kabupaten", "kecamatan", "pm25_unit", "so2_unit", "co_unit"]].head(10))


## 2. Profiling 27 file AQMS (pass 1)

Pass pertama membaca setiap file dan menghitung:
- inventory baris/kolom/satuan,
- distribusi `qc_flag`,
- jumlah blank, teks error, sentinel/out-of-range,
- mean dan batas IQR dari nilai VALID untuk imputasi/outlier.

Deteksi sentinel hanya diterapkan pada baris non-VALID. Nilai ekstrem pada baris VALID (mis. episode kabut asap gambut) tetap dianggap pengukuran nyata.


In [ ]:
def normalize_unit(u):
    u = str(u).strip().lower()
    if u in ("ppm", "ppb"):
        return "ppm"
    if u in ("ug/m3", "µg/m³", "ug_m3", "ugm3", "ug/m³"):
        return "ug_m3"
    return "unknown"


def process_numeric(orig, qc, bounds):
    s0 = pd.to_numeric(orig, errors="coerce")
    text = orig.notna() & s0.isna()
    inf = pd.Series(np.isinf(s0), index=s0.index)
    s = s0.mask(inf)
    lo, hi = bounds
    out = (s < lo) | (s > hi)
    invalid = text | inf | ((qc != "VALID") & out)
    return s, text, inf, invalid


def ppm_to_ugm3(ppm, mw, temp_c):
    temp = pd.to_numeric(temp_c, errors="coerce")
    temp = temp.mask(~np.isfinite(temp), np.nan)
    temp_use = temp.where(temp.notna() & temp.between(-20, 60), TEMP_FALLBACK_C)
    vm = R_ATM_L_PER_MOL_K * (temp_use + 273.15) / P_ATM
    return ppm * mw * 1000.0 / vm


def iqr_bounds(series):
    s = series.dropna()
    n = len(s)
    if n < 5:
        return (np.nan, np.nan, np.nan, np.nan, np.nan, n)
    q1 = float(s.quantile(0.25))
    q3 = float(s.quantile(0.75))
    iqr = q3 - q1
    return (q1, q3, iqr, q1 - 1.5 * iqr, q3 + 1.5 * iqr, n)


inventory_parts = []
qc_parts = []
stat_rows = []
stat_lookup = {}
file_meta = {}

global_sum = {p: 0.0 for p in POLLUTANTS}
global_cnt = {p: 0 for p in POLLUTANTS}
global_sum_met = {c: 0.0 for c in METEO_COLS}
global_cnt_met = {c: 0 for c in METEO_COLS}

aqms_min_ts = None
aqms_max_ts = None

for f in AQMS_FILES:
    d = pd.read_csv(f, dtype=str)
    d["qc_flag"] = d["qc_flag"].fillna("UNKNOWN").astype(str).str.strip()
    qc = d["qc_flag"]

    sensor_id = str(d["sensor_id"].iloc[0])
    loc_id = sensor_to_location.get(sensor_id, "UNKNOWN")
    site_row = site_by_loc.get(loc_id, {})

    units = {}
    for p in POLLUTANTS:
        u = normalize_unit(site_row.get(f"{p}_unit", ""))
        if u == "unknown":
            u = "ppm" if f"{p}_ppm" in d.columns else "ug_m3"
        units[p] = u
    unit_mode = "ppm" if any(units[p] == "ppm" for p in POLLUTANTS) else "ug_m3"
    file_meta[f.name] = {
        "sensor_id": sensor_id,
        "location_id": loc_id,
        "units": units,
        "unit_mode": unit_mode,
    }

    inventory_parts.append({
        "source_file": f.name,
        "rows": len(d),
        "columns": len(d.columns),
        "measurement_unit": unit_mode,
        "sensor_id": sensor_id,
        "location_id": loc_id,
    })

    for val, cnt in qc.value_counts().items():
        qc_parts.append({
            "source_file": f.name,
            "qc_flag": val,
            "count": int(cnt),
            "pct": round(100 * int(cnt) / len(d), 2),
        })

    ts = pd.to_datetime(d["timestamp_start"], errors="coerce")
    valid_ts = ts.notna() & (ts >= pd.Timestamp("2000-01-01")) & (ts <= pd.Timestamp("2030-12-31"))
    if valid_ts.any():
        mn, mx = ts[valid_ts].min(), ts[valid_ts].max()
        aqms_min_ts = mn if aqms_min_ts is None else min(aqms_min_ts, mn)
        aqms_max_ts = mx if aqms_max_ts is None else max(aqms_max_ts, mx)

    for c in METEO_COLS:
        orig = d[c] if c in d.columns else pd.Series(np.nan, index=d.index, dtype=object)
        s, text, inf, invalid = process_numeric(orig, qc, PLAUSIBLE_BOUNDS[c])
        valid = (qc == "VALID") & (~invalid) & s.notna()
        target = s.where(~invalid & s.notna())
        mean = float(target[valid].mean()) if valid.any() else np.nan
        q1, q3, iqr, lower, upper, n_used = iqr_bounds(target[valid])

        stat_lookup[(f.name, c)] = {
            "mean": mean,
            "n_valid": int(valid.sum()),
            "blank": int(orig.isna().sum()),
            "text": int(text.sum()),
            "sentinel": int((invalid & ~text).sum()),
            "q1": q1, "q3": q3, "iqr": iqr,
            "lower": lower, "upper": upper, "n_used": n_used,
            "unit": "native", "target": c,
        }
        stat_rows.append({
            "source_file": f.name,
            "column": c,
            "unit": "native",
            "blank": int(orig.isna().sum()),
            "text_error": int(text.sum()),
            "invalid_code": int((invalid & ~text).sum()),
            "valid_n": int(valid.sum()),
        })
        if valid.any():
            global_sum_met[c] += float(target[valid].sum())
            global_cnt_met[c] += int(valid.sum())

    for p in POLLUTANTS:
        u = units[p]
        src = f"{p}_ppm" if u == "ppm" else f"{p}_ug_m3"
        orig = d[src] if src in d.columns else pd.Series(np.nan, index=d.index, dtype=object)
        bounds = PPM_BOUNDS[p] if u == "ppm" else PLAUSIBLE_BOUNDS[p]
        s, text, inf, invalid = process_numeric(orig, qc, bounds)

        if u == "ppm" and p in GAS_PARAMS:
            target = ppm_to_ugm3(s, MOLECULAR_WEIGHT[p], d["temperature_c"])
            target = target.where(~invalid & s.notna())
        elif u == "ppm" and p in PM_PARAMS:
            target = pd.Series(np.nan, index=d.index, dtype="float64")
        else:
            target = s.where(~invalid & s.notna())

        valid = (qc == "VALID") & (~invalid) & s.notna()
        mean = float(target[valid].mean()) if valid.any() and target[valid].notna().any() else np.nan
        q1, q3, iqr, lower, upper, n_used = iqr_bounds(target[valid])

        stat_lookup[(f.name, p)] = {
            "mean": mean,
            "n_valid": int(target[valid].notna().sum()),
            "blank": int(orig.isna().sum()),
            "text": int(text.sum()),
            "sentinel": int((invalid & ~text).sum()),
            "q1": q1, "q3": q3, "iqr": iqr,
            "lower": lower, "upper": upper, "n_used": n_used,
            "unit": u, "target": "ug_m3",
        }
        stat_rows.append({
            "source_file": f.name,
            "column": p,
            "unit": u,
            "blank": int(orig.isna().sum()),
            "text_error": int(text.sum()),
            "invalid_code": int((invalid & ~text).sum()),
            "valid_n": int(target[valid].notna().sum()),
        })
        if valid.any() and target[valid].notna().any():
            global_sum[p] += float(target[valid].sum())
            global_cnt[p] += int(target[valid].notna().sum())

global_mean = {p: (global_sum[p] / global_cnt[p] if global_cnt[p] else np.nan) for p in POLLUTANTS}
global_mean_met = {c: (global_sum_met[c] / global_cnt_met[c] if global_cnt_met[c] else np.nan) for c in METEO_COLS}

inventory_df = pd.DataFrame(inventory_parts)
qc_df = pd.DataFrame(qc_parts)
source_stats_df = pd.DataFrame(stat_rows)

print("Total baris AQMS :", int(inventory_df["rows"].sum()))
print("Rentang AQMS     :", aqms_min_ts, "->", aqms_max_ts)
print("Global mean      :", {k: round(v, 3) for k, v in global_mean.items()})


### 2.1 Ringkasan profiling

Tabel berikut adalah hasil pembacaan langsung sebelum transformasi.


In [ ]:
display(Markdown("**Inventory 27 file AQMS**"))
display(inventory_df)

display(Markdown("**Distribusi qc_flag per stasiun**"))
qc_pivot = qc_df.pivot_table(index="source_file", columns="qc_flag", values="count", fill_value=0)
display(qc_pivot)

display(Markdown("**Blank, teks error, sentinel/out-of-range, dan nilai VALID per file (total parameter)**"))
prof_summary = (
    source_stats_df.groupby("source_file")[["blank", "text_error", "invalid_code", "valid_n"]]
    .sum()
    .sort_values("invalid_code", ascending=False)
)
display(prof_summary)

display(Markdown("**Kategori qc_flag unik (dipertahankan seluruhnya)**"))
print(sorted(qc_df["qc_flag"].unique().tolist()))


## 3. Dimensi tanggal dan waktu

`dim_date` mencakup seluruh periode AQMS dan Health Outcomes (2023-12-31 s.d. 2026-10-14).  
`dim_time` memiliki 96 kunci 15 menit sehingga mendukung resolusi 15 menit dan 1 jam.


In [ ]:
health_dates = pd.to_datetime(health_raw["date"], errors="coerce")
start_candidates = [x for x in [health_dates.min(), aqms_min_ts] if pd.notna(x)]
end_candidates = [x for x in [health_dates.max(), aqms_max_ts] if pd.notna(x)]
start_date = min(start_candidates).normalize()
end_date = max(end_candidates).normalize()
dates = pd.date_range(start_date, end_date, freq="D")

dim_date = pd.DataFrame({
    "date_key": dates.strftime("%Y%m%d").astype(int),
    "date": dates.strftime("%Y-%m-%d"),
    "year": dates.year,
    "quarter": dates.quarter,
    "month": dates.month,
    "month_name": dates.month_name(),
    "day": dates.day,
    "day_name": dates.day_name(),
    "day_of_week": dates.dayofweek + 1,
    "is_weekend": dates.dayofweek >= 5,
})

time_rows = []
for h in range(24):
    for m in (0, 15, 30, 45):
        time_rows.append({
            "time_key": h * 100 + m,
            "hour": h,
            "minute": m,
            "time_label": f"{h:02d}:{m:02d}",
            "is_hourly": m == 0,
        })
dim_time = pd.DataFrame(time_rows)

print("dim_date:", dim_date.shape, dim_date["date"].min(), "->", dim_date["date"].max())
print("dim_time:", dim_time.shape)
display(dim_date.head())
display(dim_time.head(8))


## 4. Dimensi site, sensor, dan hospital

- `dim_site` berasal langsung dari `sites.csv`.
- `dim_sensor` berasal dari `sensors.csv`; relasi ke lokasi tetap melalui `location_id`.
- `dim_hospital` berasal dari `hospitals.csv`; **tidak ada latitude/longitude** yang ditambahkan.


In [ ]:
dim_site = sites.copy()
dim_site.insert(0, "site_key", dim_site["location_id"])
for c in ["latitude_dd", "longitude_dd", "baseline_pm25", "distance_to_road_m", "traffic_intensity_per_h"]:
    dim_site[c] = pd.to_numeric(dim_site[c], errors="coerce")

dim_sensor = sensors[["sensor_id", "location_id"] + [c for c in sensors.columns if c not in ("sensor_id", "location_id")]].copy()
for c in ["latitude", "longitude"]:
    dim_sensor[c] = pd.to_numeric(dim_sensor[c], errors="coerce")

dim_hospital = hospitals[["hospital_id"] + [c for c in hospitals.columns if c != "hospital_id"]].copy()
dim_hospital["kapasitas_tempat_tidur"] = pd.to_numeric(dim_hospital["kapasitas_tempat_tidur"], errors="coerce")

print("dim_site:", dim_site.shape)
print("dim_sensor:", dim_sensor.shape)
print("dim_hospital:", dim_hospital.shape)
print("Hospital memiliki kolom koordinat?", any(c in dim_hospital.columns for c in ["latitude", "longitude", "latitude_dd", "longitude_dd"]))
display(dim_site.head(3))
display(dim_sensor.head(3))
display(dim_hospital.head(3))


## 5. Dokumentasi standardisasi satuan

Aturan:
- Stasiun µg/m³: tidak ada konversi (`valid_native`).
- Gas ppm: konversi per baris menggunakan MW dan suhu baris; tekanan 1 atm diasumsikan.
- PM ppm: **tidak dikonversi** (`unresolved`), kandidat x1000 hanya sebagai asumsi.


In [ ]:
unit_doc_parts = []
for f in AQMS_FILES:
    station_key = f.name.replace("_AQMS.csv", "")
    meta = file_meta[f.name]
    for p in POLLUTANTS:
        u = meta["units"][p]
        base = {
            "station_key": station_key,
            "location_id": meta["location_id"],
            "sensor_id": meta["sensor_id"],
            "parameter": p,
            "source_column": f"{p}_ppm" if u == "ppm" else f"{p}_ug_m3",
            "unit_original": u,
        }
        if u == "ug_m3":
            base.update({
                "unit_target": "ug/m3",
                "molecular_weight_g_mol": np.nan,
                "conversion_factor_reference": np.nan,
                "formula": "Sudah dalam ug/m3; tidak ada konversi",
                "temperature_basis": "-",
                "pressure_assumption": "-",
                "status": "valid_native",
                "candidate_factor": np.nan,
                "candidate_status": "not_applicable",
                "notes": "",
            })
        elif p in GAS_PARAMS:
            mw = MOLECULAR_WEIGHT[p]
            factor25 = mw * 1000.0 / 24.465
            base.update({
                "unit_target": "ug/m3",
                "molecular_weight_g_mol": mw,
                "conversion_factor_reference": round(factor25, 4),
                "formula": "ug/m3 = ppm * MW * 1000 / (0.082057366 * T_K / 1 atm)",
                "temperature_basis": "temperature_c per baris; fallback 25 C",
                "pressure_assumption": "1 atm (asumsi; tekanan tidak tersedia)",
                "status": "valid_converted",
                "candidate_factor": np.nan,
                "candidate_status": "not_applicable",
                "notes": "Faktor referensi dihitung pada 25 C",
            })
        else:
            base.update({
                "unit_target": "ug/m3 (unresolved)",
                "molecular_weight_g_mol": np.nan,
                "conversion_factor_reference": np.nan,
                "formula": "Tidak dikonversi (PM bukan gas)",
                "temperature_basis": "-",
                "pressure_assumption": "-",
                "status": "unresolved",
                "candidate_factor": PM_CANDIDATE_FACTOR,
                "candidate_status": "assumption_only",
                "notes": "Kandidat: 1 ppm diasumsikan 1 mg/m3 lalu x1000; bukan nilai resmi",
            })
        unit_doc_parts.append(base)

unit_conversion_doc = pd.DataFrame(unit_doc_parts)
print("unit_conversion_documentation:", unit_conversion_doc.shape)
display(unit_conversion_doc.groupby(["unit_original", "status"]).size().reset_index(name="n"))
display(unit_conversion_doc[unit_conversion_doc["status"] == "unresolved"].head(12))


## 6. Pembentukan FACT_AIR_QUALITY (pass 2)

Grain: **satu pengukuran sensor pada satu waktu**. Seluruh 27 file di-append.  
Kolom kunci: `measurement_id`, `sensor_id`, `site_key`, `date_key`, `time_key`, `qc_flag`.  
Kolom nilai mempertahankan raw, hasil standardisasi, hasil imputasi, dan flag kualitas/outlier.


In [ ]:
FACT_BASE_COLS = [
    "measurement_id", "sensor_id", "site_key", "source_file",
    "timestamp_start", "timestamp_end", "date_key", "time_key",
    "time_resolution", "qc_flag", "is_qc_valid", "is_timestamp_valid",
    "data_quality_status", "measurement_unit",
]


def pollutant_fact_cols(p):
    cols = [f"{p}_raw", f"{p}_ug_m3", f"{p}_unit_status"]
    if p in PM_PARAMS:
        cols.append(f"{p}_candidate_ug_m3")
    cols += [f"{p}_is_invalid_code", f"{p}_imputed", f"{p}_was_imputed", f"{p}_is_outlier"]
    return cols


def meteo_fact_cols(c):
    return [c, f"{c}_is_invalid_code", f"{c}_imputed", f"{c}_was_imputed", f"{c}_is_outlier"]


FACT_COLS = (
    FACT_BASE_COLS
    + [col for p in POLLUTANTS for col in pollutant_fact_cols(p)]
    + [col for c in METEO_COLS for col in meteo_fact_cols(c)]
)

fact_parts = []
missing_parts = []
impute_parts = []
outlier_parts = []
issue_parts = []

for f in AQMS_FILES:
    d = pd.read_csv(f, dtype=str)
    d["qc_flag"] = d["qc_flag"].fillna("UNKNOWN").astype(str).str.strip()
    qc = d["qc_flag"]

    meta = file_meta[f.name]
    sensor_id = meta["sensor_id"]
    loc_id = meta["location_id"]
    units = meta["units"]
    unit_mode = meta["unit_mode"]

    ts = pd.to_datetime(d["timestamp_start"], errors="coerce")
    ts_end = pd.to_datetime(d["timestamp_end"], errors="coerce")
    valid_ts = ts.notna() & (ts >= pd.Timestamp("2000-01-01")) & (ts <= pd.Timestamp("2030-12-31"))

    date_key = pd.Series(pd.NA, index=d.index, dtype="Int64")
    date_key.loc[valid_ts] = pd.to_numeric(
        ts[valid_ts].dt.strftime("%Y%m%d"), errors="coerce"
    ).astype("Int64")

    hour = ts.dt.hour
    minute = ts.dt.minute
    quarter = minute.isin([0, 15, 30, 45])
    time_key = pd.Series(pd.NA, index=d.index, dtype="Int64")
    time_key.loc[valid_ts] = (hour[valid_ts] * 100).astype("Int64")
    q_mask = valid_ts & quarter
    time_key.loc[q_mask] = (hour[q_mask] * 100 + minute[q_mask]).astype("Int64")

    time_resolution = pd.Series("invalid", index=d.index, dtype=object)
    time_resolution.loc[valid_ts] = "hourly"
    time_resolution.loc[valid_ts & quarter & (minute != 0)] = "15-minute"

    pq = {}
    mq = {}

    for c in METEO_COLS:
        orig = d[c] if c in d.columns else pd.Series(np.nan, index=d.index, dtype=object)
        s, text, inf, invalid = process_numeric(orig, qc, PLAUSIBLE_BOUNDS[c])
        target = s.where(~invalid & s.notna())
        valid = (qc == "VALID") & (~invalid) & s.notna()

        stat = stat_lookup[(f.name, c)]
        mean = stat["mean"]
        fallback = False
        if pd.isna(mean) and pd.notna(global_mean_met[c]):
            mean = global_mean_met[c]
            fallback = True
        blank = orig.isna()
        was = blank & target.isna() & pd.notna(mean)
        imputed = target.copy()
        imputed.loc[was] = mean

        lo_b, hi_b = stat["lower"], stat["upper"]
        if pd.notna(lo_b) and pd.notna(hi_b):
            is_outlier = imputed.notna() & ((imputed < lo_b) | (imputed > hi_b))
        else:
            is_outlier = pd.Series(False, index=d.index)

        mq[c] = {
            "orig_blank": blank, "s": s, "text": text, "inf": inf,
            "invalid": invalid, "target": target, "imputed": imputed,
            "was": was, "is_outlier": is_outlier, "mean": mean,
        }

        missing_parts.append({
            "table": "aqms", "source_file": f.name, "sensor_id": sensor_id,
            "location_id": loc_id, "parameter": c, "source_column": c,
            "unit_original": "native", "total_rows": len(d),
            "blank": int(blank.sum()), "text_error": int(text.sum()),
            "invalid_code": int((invalid & ~text).sum()),
            "imputed_n": int(was.sum()), "valid_n": int(valid.sum()),
            "valid_target_n": int(target[valid].notna().sum()),
            "unresolved_unit": False,
        })

        if int(blank.sum()) > 0:
            impute_parts.append({
                "table": "aqms", "source_file": f.name, "sensor_id": sensor_id,
                "location_id": loc_id, "parameter": c, "source_column": c,
                "unit_target": "native", "blank_n": int(blank.sum()),
                "imputed_n": int(was.sum()), "impute_mean": mean,
                "method": "mean_sensor_parameter_valid",
                "fallback_used": bool(fallback),
                "status": "imputed" if bool(was.any()) else "no_imputation",
                "notes": "",
            })

        outlier_parts.append({
            "table": "aqms", "source_file": f.name, "sensor_id": sensor_id,
            "location_id": loc_id, "parameter": c, "unit": "native",
            "method": "IQR_1.5", "n_used": stat["n_used"],
            "q1": stat["q1"], "q3": stat["q3"], "iqr": stat["iqr"],
            "lower_bound": lo_b, "upper_bound": hi_b,
            "outlier_count": int(is_outlier.sum()),
            "outlier_pct": round(100 * int(is_outlier.sum()) / len(d), 4),
            "total_rows": len(d),
            "status": "assessed" if pd.notna(lo_b) else "not_enough_valid_data",
        })

        if invalid.any():
            issue_parts.append(pd.DataFrame({
                "measurement_id": d.loc[invalid, "measurement_id"].astype(str).values,
                "sensor_id": d.loc[invalid, "sensor_id"].astype(str).values,
                "source_file": f.name,
                "timestamp_start": d.loc[invalid, "timestamp_start"].values,
                "column": c,
                "raw_value": orig.loc[invalid].astype(str).values,
                "issue_type": np.where(
                    text.loc[invalid].values, "text_error",
                    np.where(inf.loc[invalid].values, "non_finite", "sentinel_or_out_of_range"),
                ),
                "qc_flag": qc.loc[invalid].values,
            }))

    for p in POLLUTANTS:
        u = units[p]
        src = f"{p}_ppm" if u == "ppm" else f"{p}_ug_m3"
        orig = d[src] if src in d.columns else pd.Series(np.nan, index=d.index, dtype=object)
        bounds = PPM_BOUNDS[p] if u == "ppm" else PLAUSIBLE_BOUNDS[p]
        s, text, inf, invalid = process_numeric(orig, qc, bounds)

        if u == "ppm" and p in GAS_PARAMS:
            target = ppm_to_ugm3(s, MOLECULAR_WEIGHT[p], d["temperature_c"])
            target = target.where(~invalid & s.notna())
            unit_status = "valid_converted"
            unresolved = False
        elif u == "ppm" and p in PM_PARAMS:
            target = pd.Series(np.nan, index=d.index, dtype="float64")
            unit_status = "unresolved"
            unresolved = True
        else:
            target = s.where(~invalid & s.notna())
            unit_status = "valid_native"
            unresolved = False

        valid = (qc == "VALID") & (~invalid) & s.notna()
        stat = stat_lookup[(f.name, p)]

        if unresolved:
            mean = np.nan
            fallback = False
            blank = orig.isna()
            was = pd.Series(False, index=d.index)
            imputed = target.copy()
        else:
            mean = stat["mean"]
            fallback = False
            if pd.isna(mean) and pd.notna(global_mean[p]):
                mean = global_mean[p]
                fallback = True
            blank = orig.isna()
            was = blank & target.isna() & pd.notna(mean)
            imputed = target.copy()
            imputed.loc[was] = mean

        if u == "ppm" and p in PM_PARAMS:
            candidate = (s * PM_CANDIDATE_FACTOR).where(~invalid & s.notna())
        else:
            candidate = pd.Series(np.nan, index=d.index, dtype="float64")

        lo_b, hi_b = stat["lower"], stat["upper"]
        if pd.notna(lo_b) and pd.notna(hi_b):
            is_outlier = imputed.notna() & ((imputed < lo_b) | (imputed > hi_b))
        else:
            is_outlier = pd.Series(False, index=d.index)

        pq[p] = {
            "orig_blank": blank, "s": s, "text": text, "inf": inf,
            "invalid": invalid, "target": target, "imputed": imputed,
            "was": was, "is_outlier": is_outlier, "candidate": candidate,
            "unit_status": unit_status, "mean": mean,
        }

        missing_parts.append({
            "table": "aqms", "source_file": f.name, "sensor_id": sensor_id,
            "location_id": loc_id, "parameter": p, "source_column": src,
            "unit_original": u, "total_rows": len(d),
            "blank": int(blank.sum()), "text_error": int(text.sum()),
            "invalid_code": int((invalid & ~text).sum()),
            "imputed_n": int(was.sum()), "valid_n": int(valid.sum()),
            "valid_target_n": int(target[valid].notna().sum()),
            "unresolved_unit": bool(unresolved),
        })

        if int(blank.sum()) > 0 or unresolved:
            impute_parts.append({
                "table": "aqms", "source_file": f.name, "sensor_id": sensor_id,
                "location_id": loc_id, "parameter": p, "source_column": src,
                "unit_target": "ug/m3", "blank_n": int(blank.sum()),
                "imputed_n": int(was.sum()), "impute_mean": mean,
                "method": "mean_sensor_parameter_valid",
                "fallback_used": bool(fallback),
                "status": "unresolved_not_imputed" if unresolved else ("imputed" if bool(was.any()) else "no_imputation"),
                "notes": "PM ppm unresolved; tidak dikonversi" if unresolved else "",
            })

        outlier_parts.append({
            "table": "aqms", "source_file": f.name, "sensor_id": sensor_id,
            "location_id": loc_id, "parameter": p, "unit": "ug/m3",
            "method": "IQR_1.5", "n_used": stat["n_used"],
            "q1": stat["q1"], "q3": stat["q3"], "iqr": stat["iqr"],
            "lower_bound": lo_b, "upper_bound": hi_b,
            "outlier_count": int(is_outlier.sum()),
            "outlier_pct": round(100 * int(is_outlier.sum()) / len(d), 4),
            "total_rows": len(d),
            "status": "not_assessed_pm_unresolved" if unresolved else ("assessed" if pd.notna(lo_b) else "not_enough_valid_data"),
        })

        if invalid.any():
            issue_parts.append(pd.DataFrame({
                "measurement_id": d.loc[invalid, "measurement_id"].astype(str).values,
                "sensor_id": d.loc[invalid, "sensor_id"].astype(str).values,
                "source_file": f.name,
                "timestamp_start": d.loc[invalid, "timestamp_start"].values,
                "column": p,
                "raw_value": orig.loc[invalid].astype(str).values,
                "issue_type": np.where(
                    text.loc[invalid].values, "text_error",
                    np.where(inf.loc[invalid].values, "non_finite", "sentinel_or_out_of_range"),
                ),
                "qc_flag": qc.loc[invalid].values,
            }))

    all_blank = pd.concat(
        [pq[p]["orig_blank"] for p in POLLUTANTS] + [mq[c]["orig_blank"] for c in METEO_COLS],
        axis=1,
    ).all(axis=1)
    any_text = pd.concat(
        [pq[p]["text"] for p in POLLUTANTS] + [mq[c]["text"] for c in METEO_COLS],
        axis=1,
    ).any(axis=1)
    any_invalid = pd.concat(
        [pq[p]["invalid"] for p in POLLUTANTS] + [mq[c]["invalid"] for c in METEO_COLS],
        axis=1,
    ).any(axis=1)

    dq_status = pd.Series("valid", index=d.index, dtype=object)
    dq_status.loc[qc != "VALID"] = "qc_nonvalid"
    dq_status.loc[all_blank] = "missing"
    dq_status.loc[any_invalid] = "sentinel_or_invalid"
    dq_status.loc[any_text] = "error_text"
    dq_status.loc[~valid_ts] = "invalid_timestamp"

    if (~valid_ts).any():
        issue_parts.append(pd.DataFrame({
            "measurement_id": d.loc[~valid_ts, "measurement_id"].astype(str).values,
            "sensor_id": d.loc[~valid_ts, "sensor_id"].astype(str).values,
            "source_file": f.name,
            "timestamp_start": d.loc[~valid_ts, "timestamp_start"].values,
            "column": "timestamp_start",
            "raw_value": d.loc[~valid_ts, "timestamp_start"].astype(str).values,
            "issue_type": "invalid_timestamp",
            "qc_flag": qc.loc[~valid_ts].values,
        }))

    part = pd.DataFrame({
        "measurement_id": d["measurement_id"].astype(str),
        "sensor_id": d["sensor_id"].astype(str),
        "site_key": loc_id,
        "source_file": f.name,
        "timestamp_start": ts,
        "timestamp_end": ts_end,
        "date_key": date_key,
        "time_key": time_key,
        "time_resolution": time_resolution,
        "qc_flag": qc,
        "is_qc_valid": (qc == "VALID").astype(bool),
        "is_timestamp_valid": valid_ts.astype(bool),
        "data_quality_status": dq_status,
        "measurement_unit": unit_mode,
    }, index=d.index)

    for p in POLLUTANTS:
        q = pq[p]
        part[f"{p}_raw"] = q["s"]
        part[f"{p}_ug_m3"] = q["target"]
        part[f"{p}_unit_status"] = q["unit_status"]
        if p in PM_PARAMS:
            part[f"{p}_candidate_ug_m3"] = q["candidate"]
        part[f"{p}_is_invalid_code"] = q["invalid"].astype(bool)
        part[f"{p}_imputed"] = q["imputed"]
        part[f"{p}_was_imputed"] = q["was"].astype(bool)
        part[f"{p}_is_outlier"] = q["is_outlier"].astype(bool)

    for c in METEO_COLS:
        q = mq[c]
        part[c] = q["s"]
        part[f"{c}_is_invalid_code"] = q["invalid"].astype(bool)
        part[f"{c}_imputed"] = q["imputed"]
        part[f"{c}_was_imputed"] = q["was"].astype(bool)
        part[f"{c}_is_outlier"] = q["is_outlier"].astype(bool)

    fact_parts.append(part[FACT_COLS])

fact_air_quality = pd.concat(fact_parts, ignore_index=True)
fact_air_quality = fact_air_quality.sort_values(
    ["sensor_id", "timestamp_start"], na_position="last"
).reset_index(drop=True)

missing_summary = pd.DataFrame(missing_parts)
imputation_log_aqms = pd.DataFrame(impute_parts)
outlier_summary_aqms = pd.DataFrame(outlier_parts)
data_quality_issues = pd.concat(issue_parts, ignore_index=True) if issue_parts else pd.DataFrame(
    columns=["measurement_id", "sensor_id", "source_file", "timestamp_start", "column", "raw_value", "issue_type", "qc_flag"]
)

print("FACT_AIR_QUALITY:", fact_air_quality.shape)
print("Imputation log :", imputation_log_aqms.shape)
print("Outlier summary:", outlier_summary_aqms.shape)
print("Quality issues :", data_quality_issues.shape)
display(fact_air_quality.head(3))


## 7. Pembentukan FACT_HEALTH

Grain: **satu rumah sakit pada satu tanggal** (`date` + `hospital_id`), sudah unik di sumber.  
Semua 27.491 baris dipertahankan; tidak ada imputasi karena tidak ada missing. Outlier hanya ditandai.


In [ ]:
HEALTH_OUTCOMES = [
    "resp_admissions_under5", "resp_admissions_over60", "asthma_emergency_cases",
    "copd_admissions", "resp", "cvd", "death",
]
HEALTH_WEATHER = ["temp_c", "dptp_c", "rhum_pct"]
HEALTH_NUMERIC = HEALTH_WEATHER + HEALTH_OUTCOMES

h = health_raw.copy()
h["date"] = pd.to_datetime(h["date"], errors="coerce")
h["date_key"] = pd.to_numeric(h["date"].dt.strftime("%Y%m%d"), errors="coerce").astype("Int64")
h["time_hour"] = pd.to_numeric(h["time"], errors="coerce")
h["time_key"] = (h["time_hour"] * 100).astype("Int64")

for c in HEALTH_NUMERIC:
    h[c] = pd.to_numeric(h[c], errors="coerce")

h = h.sort_values(["hospital_id", "date"]).reset_index(drop=True)
h.insert(0, "health_record_id", [f"HLTH-{i:06d}" for i in range(1, len(h) + 1)])

fact_health = h[[
    "health_record_id", "date_key", "hospital_id", "date",
    "time_key", "time_hour", "dow",
] + HEALTH_NUMERIC].copy()

fact_health["data_quality_status"] = "valid"
fact_health["has_missing_outcome"] = fact_health[HEALTH_OUTCOMES].isna().any(axis=1)

health_outlier_parts = []
for c in HEALTH_OUTCOMES:
    for hid, grp in fact_health.groupby("hospital_id"):
        lo_bound, hi_bound = np.nan, np.nan
        n_used = int(grp[c].notna().sum())
        if n_used >= 5:
            q1 = float(grp[c].quantile(0.25))
            q3 = float(grp[c].quantile(0.75))
            iqr = q3 - q1
            lo_bound, hi_bound = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        health_outlier_parts.append({
            "hospital_id": hid, "parameter": c, "method": "IQR_1.5",
            "n_used": n_used, "q1": q1 if n_used >= 5 else np.nan,
            "q3": q3 if n_used >= 5 else np.nan,
            "iqr": iqr if n_used >= 5 else np.nan,
            "lower_bound": lo_bound, "upper_bound": hi_bound,
        })
health_outlier_bounds = pd.DataFrame(health_outlier_parts)

for c in HEALTH_OUTCOMES:
    bounds = health_outlier_bounds[health_outlier_bounds["parameter"] == c].set_index("hospital_id")
    lo = fact_health["hospital_id"].map(bounds["lower_bound"])
    hi = fact_health["hospital_id"].map(bounds["upper_bound"])
    fact_health[f"{c}_is_outlier"] = (
        fact_health[c].notna() & lo.notna() & hi.notna()
        & ((fact_health[c] < lo) | (fact_health[c] > hi))
    )

health_outlier_counts = pd.DataFrame([{
    "parameter": c,
    "outlier_count": int(fact_health[f"{c}_is_outlier"].sum()),
    "outlier_pct": round(100 * int(fact_health[f"{c}_is_outlier"].sum()) / len(fact_health), 4),
    "n_used": int(fact_health[c].notna().sum()),
} for c in HEALTH_OUTCOMES])

print("FACT_HEALTH:", fact_health.shape)
print("Rentang tanggal:", fact_health["date"].min(), "->", fact_health["date"].max())
print("Jumlah RS unik :", fact_health["hospital_id"].nunique())
print("Duplikat date+hospital:", int(fact_health.duplicated(["date", "hospital_id"]).sum()))
display(fact_health.head(3))
display(health_outlier_counts)


## 8. Ringkasan dimensi dan fact

Tabel di bawah menjadi dasar pemeriksaan ukuran dan kunci sebelum ekspor.


In [ ]:
dimension_summary = pd.DataFrame([
    {"table": "dim_date", "rows": len(dim_date), "primary_key": "date_key"},
    {"table": "dim_time", "rows": len(dim_time), "primary_key": "time_key"},
    {"table": "dim_site", "rows": len(dim_site), "primary_key": "site_key"},
    {"table": "dim_sensor", "rows": len(dim_sensor), "primary_key": "sensor_id"},
    {"table": "dim_hospital", "rows": len(dim_hospital), "primary_key": "hospital_id"},
    {"table": "fact_air_quality", "rows": len(fact_air_quality), "primary_key": "measurement_id"},
    {"table": "fact_health", "rows": len(fact_health), "primary_key": "health_record_id"},
])

aqms_dq = fact_air_quality["data_quality_status"].value_counts(dropna=False).rename_axis("data_quality_status").reset_index(name="count")
health_dq = fact_health["data_quality_status"].value_counts(dropna=False).rename_axis("data_quality_status").reset_index(name="count")

display(dimension_summary)
display(Markdown("**Distribusi data_quality_status AQMS**"))
display(aqms_dq)
display(Markdown("**Distribusi data_quality_status Health**"))
display(health_dq)


## 9. Validasi kualitas dan integritas model

Validasi mencakup ukuran baris, keunikan primary key, foreign key, konsistensi grain, dan kelengkapan hasil transformasi. Notebook tidak melanjutkan ekspor jika ada pemeriksaan `FAIL`.


In [ ]:
validation = []


def add_check(name, passed, details):
    validation.append({
        "check": name,
        "status": "PASS" if bool(passed) else "FAIL",
        "details": str(details),
    })


add_check("AQMS row count", len(fact_air_quality) == 667981, f"{len(fact_air_quality)} rows")
add_check("AQMS measurement_id unique", not fact_air_quality["measurement_id"].duplicated().any(),
          f"{int(fact_air_quality['measurement_id'].duplicated().sum())} duplicates")
add_check("AQMS measurement_id not null", fact_air_quality["measurement_id"].notna().all(),
          f"{int(fact_air_quality['measurement_id'].isna().sum())} null")
add_check("Health row count", len(fact_health) == 27491, f"{len(fact_health)} rows")
add_check("Health grain unique", not fact_health.duplicated(["date", "hospital_id"]).any(),
          f"{int(fact_health.duplicated(['date', 'hospital_id']).sum())} duplicates")

site_keys = set(dim_site["site_key"])
sensor_keys = set(dim_sensor["sensor_id"])
date_keys = set(dim_date["date_key"].astype(int))
time_keys = set(dim_time["time_key"].astype(int))
hospital_keys = set(dim_hospital["hospital_id"])

aq_site = fact_air_quality["site_key"].dropna().astype(str)
aq_sensor = fact_air_quality["sensor_id"].dropna().astype(str)
aq_date = fact_air_quality["date_key"].dropna().astype(int)
aq_time = fact_air_quality["time_key"].dropna().astype(int)

add_check("FK site -> dim_site", set(aq_site).issubset(site_keys),
          f"{len(set(aq_site) - site_keys)} unknown")
add_check("FK sensor -> dim_sensor", set(aq_sensor).issubset(sensor_keys),
          f"{len(set(aq_sensor) - sensor_keys)} unknown")
add_check("FK date -> dim_date", set(aq_date).issubset(date_keys),
          f"{len(set(aq_date) - date_keys)} unknown")
add_check("FK time -> dim_time", set(aq_time).issubset(time_keys),
          f"{len(set(aq_time) - time_keys)} unknown")

add_check("FK hospital -> dim_hospital", set(fact_health["hospital_id"]).issubset(hospital_keys),
          f"{len(set(fact_health['hospital_id']) - hospital_keys)} unknown")
add_check("FK health date -> dim_date", set(fact_health["date_key"].dropna().astype(int)).issubset(date_keys),
          f"{len(set(fact_health['date_key'].dropna().astype(int)) - date_keys)} unknown")

invalid_ts_count = int((~fact_air_quality["is_timestamp_valid"]).sum())
add_check("AQMS date_key null only for invalid timestamps",
          int(fact_air_quality["date_key"].isna().sum()) == invalid_ts_count,
          f"null={int(fact_air_quality['date_key'].isna().sum())}; invalid_timestamp={invalid_ts_count}")
add_check("Health date_key null", int(fact_health["date_key"].isna().sum()) == 0,
          f"{int(fact_health['date_key'].isna().sum())} null")

unresolved_cols = [
    c for c in fact_air_quality.columns
    if c.endswith("_unit_status") and (fact_air_quality[c] == "unresolved").any()
]
add_check("PM ppm unresolved handled", len(unresolved_cols) > 0,
          f"columns={unresolved_cols}; count={int((fact_air_quality[unresolved_cols] == 'unresolved').sum().sum())}")

imputed_total = int(sum(
    int(fact_air_quality[c].sum()) for c in fact_air_quality.columns if c.endswith("_was_imputed")
))
impute_log_total = int(imputation_log_aqms["imputed_n"].sum())
add_check("Imputation count matches log", imputed_total == impute_log_total,
          f"flags={imputed_total}; log={impute_log_total}")

add_check("QC categories retained", fact_air_quality["qc_flag"].nunique() >= 12,
          f"{fact_air_quality['qc_flag'].nunique()} categories")

add_check("No AQMS rows dropped/restored", len(fact_air_quality) == int(inventory_df["rows"].sum()),
          f"source={int(inventory_df['rows'].sum())}; fact={len(fact_air_quality)}")

validation_report = pd.DataFrame(validation)
print(validation_report.to_string(index=False))
failed = validation_report[validation_report["status"] == "FAIL"]
print("\nJumlah FAIL:", len(failed))

if len(failed) > 0:
    raise ValueError("Validasi gagal: " + "; ".join(failed["check"].tolist()))


## 10. Ekspor ke `data/`

Semua tabel diekspor sebagai CSV UTF-8 (kompatibel Power BI) dan Parquet bila memungkinkan.  
Dokumentasi satuan, missing, imputasi, outlier, validasi, relasi, dan manifest ikut diekspor.


In [ ]:
tables = {
    "dim_date": dim_date,
    "dim_time": dim_time,
    "dim_site": dim_site,
    "dim_sensor": dim_sensor,
    "dim_hospital": dim_hospital,
    "fact_air_quality": fact_air_quality,
    "fact_health": fact_health,
}

exported = []
for name, df in tables.items():
    csv_path = OUTPUT_DIR / f"{name}.csv"
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    size_csv = csv_path.stat().st_size
    parquet_status = "not_written"
    try:
        pq_path = OUTPUT_DIR / f"{name}.parquet"
        df.to_parquet(pq_path, index=False)
        parquet_status = f"written ({pq_path.stat().st_size} bytes)"
    except Exception as exc:
        parquet_status = f"skipped: {exc}"
    exported.append({
        "table": name,
        "rows": int(len(df)),
        "columns": int(df.shape[1]),
        "csv_bytes": int(size_csv),
        "parquet_status": parquet_status,
    })

documentation_tables = {
    "validation_report": validation_report,
    "unit_conversion_documentation": unit_conversion_doc,
    "missing_value_summary": missing_summary,
    "imputation_log_aqms": imputation_log_aqms,
    "outlier_summary_aqms": outlier_summary_aqms,
    "health_outlier_bounds": health_outlier_bounds,
    "health_outlier_counts": health_outlier_counts,
    "data_quality_issues": data_quality_issues,
    "source_inventory": inventory_df,
    "qc_flag_distribution": qc_df,
    "source_profiling_summary": source_stats_df,
}
for name, df in documentation_tables.items():
    path = OUTPUT_DIR / f"{name}.csv"
    df.to_csv(path, index=False, encoding="utf-8-sig")

data_dictionary_rows = []
for name, df in tables.items():
    for col in df.columns:
        data_dictionary_rows.append({
            "table": name,
            "column": col,
            "dtype": str(df[col].dtype),
            "non_null": int(df[col].notna().sum()),
        })
data_dictionary = pd.DataFrame(data_dictionary_rows)
data_dictionary.to_csv(OUTPUT_DIR / "data_dictionary.csv", index=False, encoding="utf-8-sig")

relationships = pd.DataFrame([
    {"from_table": "fact_air_quality", "from_column": "date_key", "to_table": "dim_date", "to_column": "date_key", "cardinality": "many-to-one", "cross_filter": "single", "active": True},
    {"from_table": "fact_air_quality", "from_column": "time_key", "to_table": "dim_time", "to_column": "time_key", "cardinality": "many-to-one", "cross_filter": "single", "active": True},
    {"from_table": "fact_air_quality", "from_column": "site_key", "to_table": "dim_site", "to_column": "site_key", "cardinality": "many-to-one", "cross_filter": "single", "active": True},
    {"from_table": "fact_air_quality", "from_column": "sensor_id", "to_table": "dim_sensor", "to_column": "sensor_id", "cardinality": "many-to-one", "cross_filter": "single", "active": True},
    {"from_table": "fact_health", "from_column": "date_key", "to_table": "dim_date", "to_column": "date_key", "cardinality": "many-to-one", "cross_filter": "single", "active": True},
    {"from_table": "fact_health", "from_column": "hospital_id", "to_table": "dim_hospital", "to_column": "hospital_id", "cardinality": "many-to-one", "cross_filter": "single", "active": True},
])
relationships.to_csv(OUTPUT_DIR / "powerbi_relationships.csv", index=False, encoding="utf-8-sig")

manifest = {
    "generated_at": datetime.now().isoformat(timespec="seconds"),
    "source_directory": str(BASE_DIR),
    "output_directory": str(OUTPUT_DIR),
    "source_rows": {
        "aqms": int(inventory_df["rows"].sum()),
        "health": int(len(health_raw)),
    },
    "tables": exported,
    "documentation_tables": list(documentation_tables.keys()),
    "decisions": {
        "pm_ppm": "unresolved; candidate_ug_m3 = ppm x 1000 labeled assumption_only",
        "gas_ppm": "per-row conversion with temperature_c, pressure assumed 1 atm, fallback 25 C",
        "missing": "blank imputed by sensor+parameter mean from VALID; sentinel/text not imputed",
        "outlier": "IQR 1.5 flag only; no drop/clip/winsorization",
        "qc_flag": "all 12 categories retained; qc_flag=-1 retained",
        "bridge_site_hospital": "not created; no official booklet/pairing",
    },
    "powerbi_relationships": relationships.to_dict(orient="records"),
}
with open(OUTPUT_DIR / "manifest.json", "w", encoding="utf-8") as fh:
    json.dump(manifest, fh, ensure_ascii=False, indent=2)

readme = f"""# Output Power BI Final

Dibuat: {manifest['generated_at']}

## Tabel
- `dim_date` ({len(dim_date)} baris): kalender harian {dim_date['date'].min()} s.d. {dim_date['date'].max()}
- `dim_time` ({len(dim_time)} baris): 96 slot 15 menit
- `dim_site` ({len(dim_site)} baris)
- `dim_sensor` ({len(dim_sensor)} baris)
- `dim_hospital` ({len(dim_hospital)} baris; tanpa koordinat)
- `fact_air_quality` ({len(fact_air_quality)} baris): grain measurement_id
- `fact_health` ({len(fact_health)} baris): grain date + hospital_id

## Relasi (single direction)
- fact_air_quality[date_key] -> dim_date[date_key]
- fact_air_quality[time_key] -> dim_time[time_key]
- fact_air_quality[site_key] -> dim_site[site_key]
- fact_air_quality[sensor_id] -> dim_sensor[sensor_id]
- fact_health[date_key] -> dim_date[date_key]
- fact_health[hospital_id] -> dim_hospital[hospital_id]

## Catatan penting
- PM ppm (PSW-01, MDN-01, BJB-01) tidak dikonversi; `*_ug_m3` NULL dan kandidat x1000 berlabel asumsi.
- Gas ppm dikonversi per baris dengan suhu; tekanan 1 atm diasumsikan.
- Blank diimputasi dengan mean sensor+parameter dari data VALID dan ditandai `*_was_imputed`.
- Outlier hanya flag `*_is_outlier`; tidak ada penghapusan.
- `qc_flag` tidak dihapus; `data_quality_status` membedakan valid, nonvalid, missing, sentinel, teks error, timestamp invalid.
- Tidak ada bridge site-hospital.
"""
(OUTPUT_DIR / "README_powerbi.md").write_text(readme, encoding="utf-8")

print("Ekspor selesai.")
display(pd.DataFrame(exported))
print("\nFile di output:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(f"- {p.name} ({p.stat().st_size:,} bytes)")


## 11. Kesimpulan

- Seluruh 27 file AQMS (667.981 baris) dan Health Outcomes (27.491 baris) berhasil diproses tanpa menghapus baris.
- Satuan telah distandardisasi: gas ppm -> µg/m³ per baris; PM ppm ditandai unresolved dengan kandidat berlabel asumsi.
- Missing blank diimputasi hanya dari mean VALID per sensor/parameter dan selalu ditandai; sentinel/teks error tidak diimputasi.
- Outlier hanya diberi flag IQR; nilai ekstrem VALID tidak dihapus.
- Model akhir terdiri dari 5 dimensi dan 2 fact table dengan relasi single-direction siap Power BI.
- Seluruh dokumentasi dan laporan validasi tersedia di `data/`.
